# Step 3: Needle in a Haystack, FP16 baseline (Qwen3-4B, T4)

**Goal.** Measure where the FP16 model retrieves a hidden fact, as a grid of context length x needle depth. This is the reference for step 4 (FP8 cache) and step 5 (eviction), so it has to be reliable before anything else is compared with it.

**What the notebook produces**
- `MyDrive/kv-cache/results/needle_fp16.jsonl`: one record per trial (length, depth, variant, correct, output, time, peak memory). Written after every trial, so a Colab disconnect loses nothing and a rerun resumes.
- `MyDrive/kv-cache/results/needle_fp16_heatmap.png`: success rate per (length, depth), with k/n printed in each cell.

**Design decisions (so step 4 and 5 can reuse this)**
- Prompts are deterministic: the same `(length, depth, variant)` always gives the same prompt, so step 4 and 5 rebuild exactly the same set and we compare *paired* results.
- For one `(length, variant)` the haystack is identical at every depth, so depth is the only thing that changes along a column. The needle for a variant is the same at every length.
- Greedy decoding, thinking mode off. Any variation comes from the needle, the haystack slice, the length and the depth, not from sampling.
- Correct = the 6-digit value appears in the answer.
- `run_grid(..., make_cache=...)` takes a cache factory. Step 4 passes a quantizing cache here, step 5 an eviction cache. Nothing else changes.

**Order of cells:** setup, config, model, haystack, needles and prompt builder, generation, grid runner, smoke test, baseline gate, full grid, heatmap.

In [ ]:
%pip install -q -U transformers accelerate datasets matplotlib pandas tqdm

In [ ]:
import json, os, random, re, time, math
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "Switch the runtime to a GPU"
GPU_NAME = torch.cuda.get_device_name(0)
print(torch.__version__, GPU_NAME, f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")

# Results live on Google Drive: they survive runtime resets and are shared with the FP8 notebook
# (every notebook gets its own Colab runtime, and /content is not shared between runtimes).
from google.colab import drive
drive.mount("/content/drive")
RESULTS_DIR = "/content/drive/MyDrive/kv-cache/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

## Config
Lengths stop at 16K because of T4 memory (weights about 7.5 GiB plus cache plus prefill workspace). 32K is a stretch point for later. FP16, not bf16: the T4 has no native bf16.

In [ ]:
CFG = dict(
    model_id="Qwen/Qwen3-4B",
    lengths=[2048, 4096, 8192, 16384],   # target prompt length in tokens
    depths=[0, 25, 50, 75, 100],         # % of the haystack before the needle (0 = start, 100 = end)
    n_variants=5,                        # needle and haystack variants per (length, depth) cell
    max_new_tokens=24,
    seed=0,
    tag="fp16",
)
RESULTS_PATH = f"{RESULTS_DIR}/needle_fp16.jsonl"
SMOKE_PATH   = f"{RESULTS_DIR}/needle_smoke.jsonl"
SMOKE = dict(lengths=[1024, 2048], depths=[0, 50, 100], n_variants=2)
GATE_LENGTHS = [2048, 4096]              # short contexts: the baseline must be near 100% here
GATE_THRESHOLD = 0.95

## Model
`attn_implementation="sdpa"` uses PyTorch's fused attention; on a T4 it falls back to the memory-efficient kernel (FlashAttention needs sm80+). Check that a short generation looks sane: Qwen models can occasionally overflow in FP16.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(CFG["model_id"])
model = AutoModelForCausalLM.from_pretrained(
    CFG["model_id"], torch_dtype=torch.float16, device_map="cuda", attn_implementation="sdpa"
).eval()

mc = model.config
head_dim = getattr(mc, "head_dim", mc.hidden_size // mc.num_attention_heads)
KV_BYTES_PER_TOKEN = 2 * mc.num_hidden_layers * mc.num_key_value_heads * head_dim * 2   # FP16
weights_gib = sum(p.numel() * p.element_size() for p in model.parameters()) / 2**30
print(f"layers {mc.num_hidden_layers}, q heads {mc.num_attention_heads}, kv heads {mc.num_key_value_heads}, head_dim {head_dim}")
print(f"weights {weights_gib:.2f} GiB, KV cache {KV_BYTES_PER_TOKEN/1024:.0f} KiB/token "
      f"-> {KV_BYTES_PER_TOKEN*16384/2**30:.2f} GiB at 16K")

# sanity check
msgs = [{"role": "user", "content": "Say hello in five words."}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False,
                              return_tensors="pt", return_dict=True).to("cuda")
out = model.generate(**ids, max_new_tokens=16, do_sample=False, temperature=None, top_p=None, top_k=None)
print(repr(tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)))

## Haystack
WikiText-2 sentences (real prose, no network needed beyond the dataset download). Token counts per sentence are computed once so prompts can be built to a target length.

In [ ]:
from datasets import load_dataset

ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
raw = " ".join(t.strip() for t in ds["text"] if t.strip() and not t.strip().startswith("="))
raw = raw.replace(" @-@ ", "-").replace(" @,@ ", ",").replace(" @.@ ", ".")
SENTS = [s for s in re.split(r"(?<=[.!?])\s+", raw) if 40 <= len(s) <= 400][:40000]
SENT_TOK = [len(x) for x in tok(SENTS, add_special_tokens=False)["input_ids"]]
print(len(SENTS), "sentences,", sum(SENT_TOK), "tokens, mean", round(np.mean(SENT_TOK), 1), "tokens/sentence")

## Needles and prompt builder
Each variant has its own place name and its own random 6-digit value. The question comes after the document, which is the usual setup. `build_prompt` returns the prompt text plus metadata (the expected value and the *actual* depth in tokens).

In [ ]:
PLACES = ["Marlowe Harbor", "Quillfeather", "Brindlemoor", "Tessaly Ridge",
          "Orrin Falls", "Valdemar Bay", "Pellam Crossing", "Ashgrove Mill"]

def ntok(s):
    return len(tok(s, add_special_tokens=False)["input_ids"])

def build_prompt(target_len, depth, variant, seed=CFG["seed"], overhead=60):
    place = PLACES[variant % len(PLACES)]
    value = str(random.Random(f"val-{seed}-{variant}").randint(100000, 999999))
    needle = f"The special magic number for {place} is {value}."
    question = f"What is the special magic number for {place}?"
    budget = target_len - overhead - ntok(needle) - ntok(question)

    rng = random.Random(f"hay-{seed}-{target_len}-{variant}")       # same haystack at every depth
    start = rng.randrange(0, len(SENTS) - 2000)
    chosen, total, i = [], 0, start
    while total < budget:
        chosen.append(SENTS[i]); total += SENT_TOK[i]; i += 1

    k = round(depth / 100 * len(chosen))                             # needle goes before sentence k
    actual_depth = 100 * sum(SENT_TOK[start:start + k]) / total
    text = " ".join(chosen[:k] + [needle] + chosen[k:])
    prompt = f"{text}\n\nBased only on the text above, {question} Reply with just the number."
    return prompt, dict(value=value, actual_depth=actual_depth, place=place)

# check: the value appears exactly once (in the needle), length and depth are close to the targets
p, m = build_prompt(2048, 50, 0)
assert p.count(m["value"]) == 1
print("tokens in prompt text:", ntok(p), "| actual depth:", round(m["actual_depth"], 1), "%")

## Generation
One prompt in, one answer out. `generate()` computes logits only for the last position during prefill in recent transformers versions. If a 16K prompt runs out of memory at the prefill, that is the first thing to check (upgrade transformers, or write the prefill by hand with `logits_to_keep=1`).

`past_key_values` is the hook for step 4 and 5: pass a custom cache object there.

In [ ]:
@torch.no_grad()
def answer(prompt, max_new_tokens, cache=None):
    msgs = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    ids = tok(text, return_tensors="pt", add_special_tokens=False).input_ids.to("cuda")
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    t0 = time.perf_counter()
    kw = dict(past_key_values=cache) if cache is not None else {}
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                         temperature=None, top_p=None, top_k=None, **kw)
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    peak = torch.cuda.max_memory_allocated()
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True), ids.shape[1], dt, peak

## Trial and grid runner
`run_grid` writes one JSON line per trial and skips trials already in the file, so rerunning after a disconnect resumes. If a length runs out of memory, that trial and every longer one is recorded as `oom` and not tried again; pass `retry_oom=True` to try them again (for example after freeing memory).

In [ ]:
def run_trial(L, d, v, make_cache=None):
    prompt, meta = build_prompt(L, d, v)
    cache = make_cache() if make_cache is not None else None
    out_text, n_in, dt, peak = answer(prompt, CFG["max_new_tokens"], cache)
    correct = meta["value"] in out_text.replace(",", "").replace(" ", "")
    return dict(tag=CFG["tag"], length=L, depth=d, variant=v, n_tokens=n_in,
                actual_depth=round(meta["actual_depth"], 1), expected=meta["value"],
                output=out_text.strip(), correct=bool(correct),
                seconds=round(dt, 2), peak_gib=round(peak / 2**30, 2), status="ok")

def run_grid(lengths, depths, n_variants, path, make_cache=None, retry_oom=False):
    done = set()
    if os.path.exists(path):
        for line in open(path):
            r = json.loads(line)
            if r["status"] == "ok" or not retry_oom:
                done.add((r["length"], r["depth"], r["variant"]))
    todo = [(L, d, v) for L in lengths for d in depths for v in range(n_variants) if (L, d, v) not in done]
    oom_at = []
    with open(path, "a") as f:
        for L, d, v in tqdm(todo):
            if any(L >= x for x in oom_at):
                rec = dict(tag=CFG["tag"], length=L, depth=d, variant=v, status="oom")
            else:
                try:
                    rec = run_trial(L, d, v, make_cache)
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache(); oom_at.append(L)
                    print(f"OOM at length {L}; marking this and longer lengths as oom")
                    rec = dict(tag=CFG["tag"], length=L, depth=d, variant=v, status="oom")
            f.write(json.dumps(rec) + "\n"); f.flush()
    return pd.read_json(path, lines=True)

## Smoke test
A tiny grid first: checks the prompt really has the target length, the answers look right, and gives a time per trial to extrapolate from. Delete `needle_smoke.jsonl` in the results folder to rerun it.

In [ ]:
sm = run_grid(SMOKE["lengths"], SMOKE["depths"], SMOKE["n_variants"], SMOKE_PATH)
print(sm[["length", "n_tokens", "depth", "actual_depth", "variant", "expected", "output", "correct", "seconds", "peak_gib"]].to_string())

# extrapolate the full grid: time is roughly linear in tokens (attention makes it a bit worse, so this underestimates)
okr = sm[sm.status == "ok"]
b, a = np.polyfit(okr.n_tokens, okr.seconds, 1)
n_trials = {L: len(CFG["depths"]) * CFG["n_variants"] for L in CFG["lengths"]}
est = sum(n * (a + b * L) for L, n in n_trials.items())
print(f"\nfull grid: {sum(n_trials.values())} trials, rough estimate {est/60:.0f} min")

## Baseline gate
Run the short lengths first. **If the success rate here is below the threshold, stop.** A heatmap that is already red at 2K to 4K cannot tell FP8 loss from a model or prompt that does not retrieve. Fix the prompt wording or the model before continuing (and delete the results file, because the old trials were made with the old setup).

In [ ]:
g = run_grid(GATE_LENGTHS, CFG["depths"], CFG["n_variants"], RESULTS_PATH)
gg = g[(g.status == "ok") & (g.length.isin(GATE_LENGTHS))]
rate = gg.correct.mean()
print(gg.groupby("length").correct.agg(["mean", "size"]))
print(f"\nbaseline success at {GATE_LENGTHS}: {rate:.2f}  ->", "PASS" if rate >= GATE_THRESHOLD else "FAIL: do not continue")

## Full grid
Resumes where it stopped. Longest lengths take the most time and memory.

In [ ]:
df = run_grid(CFG["lengths"], CFG["depths"], CFG["n_variants"], RESULTS_PATH)
print(df.status.value_counts().to_string())

## Results
Per cell: successes / trials. The standard error of a rate p over n trials is sqrt(p(1-p)/n), for example about 0.13 for p = 0.8 and n = 10, so a cell with few trials cannot support small differences. Step 4 will compare the same cells *pairwise* (same prompt, FP16 vs FP8).

In [ ]:
def summarize(df):
    ok = df[df.status == "ok"]
    return ok.groupby(["length", "depth"])["correct"].agg(k="sum", n="size").reset_index()

def plot_heatmap(df, title, fname):
    g = summarize(df)
    lengths, depths = sorted(g.length.unique()), sorted(g.depth.unique())
    M = np.full((len(depths), len(lengths)), np.nan)
    for r in g.itertuples():
        M[depths.index(r.depth), lengths.index(r.length)] = r.k / r.n
    fig, ax = plt.subplots(figsize=(1.5 * len(lengths) + 2, 0.7 * len(depths) + 1.8))
    im = ax.imshow(M, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(len(lengths))); ax.set_xticklabels([f"{L//1024}K" for L in lengths])
    ax.set_yticks(range(len(depths)));  ax.set_yticklabels([f"{d}%" for d in depths])
    ax.set_xlabel("context length (tokens)"); ax.set_ylabel("needle depth")
    for r in g.itertuples():
        ax.text(lengths.index(r.length), depths.index(r.depth), f"{int(r.k)}/{int(r.n)}",
                ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=ax, label="success rate")
    ax.set_title(title, fontsize=10)
    fig.savefig(fname, dpi=160, bbox_inches="tight"); plt.show()

plot_heatmap(df, f"Needle in a haystack, FP16 cache, {CFG['model_id']}, {GPU_NAME}", f"{RESULTS_DIR}/needle_fp16_heatmap.png")

s = summarize(df).groupby("length").agg(k=("k", "sum"), n=("n", "sum"))
s["rate"] = s.k / s.n
s["se"] = np.sqrt(s.rate * (1 - s.rate) / s.n)
print(s.round(3).to_string())

## Download (optional)
The results are already on Drive. This also zips them and downloads a copy to your computer, and flushes pending Drive writes.

In [ ]:
import shutil
shutil.make_archive("/content/needle_fp16_results", "zip", RESULTS_DIR)
try:
    from google.colab import files
    files.download("/content/needle_fp16_results.zip")
    drive.flush_and_unmount()
except ImportError:
    print("Not on Colab; results are in", RESULTS_DIR)

## For step 4 and 5
- Same `build_prompt`, same `CFG`, same grid. Only `make_cache` and `CFG["tag"]` change, and results go to a different file (`needle_fp8.jsonl`, `needle_evict.jsonl`).
- Compare per cell: merge the two result files on `(length, depth, variant)` and count the trials where FP16 is correct and the other is wrong, and the reverse.
- If this baseline is not near 100% at short lengths, the later comparison is not interpretable. Check that first.
- Record in the report: model, dtype, GPU, trials per cell, prompt wording, and that FP8 is simulated on the T4.